<a href="https://colab.research.google.com/github/Mogi21/nlp-engineering-portfolio/blob/main/02_rag_qa_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q python-terrier pyterrier-t5 transformers IR-measures

In [8]:
import os
import pandas as pd
import pyterrier as pt
from pyterrier_t5 import MonoT5ReRanker
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# ==========================================
# 1. INITIALIZATION
# ==========================================
if not pt.started():
    pt.init()

# ==========================================
# 2. FIRST-STAGE RETRIEVAL & RE-RANKING
# ==========================================
print("Loading Wikipedia Sparse Index...")
dataset = pt.get_dataset("irds:vaswani")
sparse_index = pt.get_dataset("vaswani").get_index()
bm25_ret = pt.BatchRetrieve(sparse_index, wmodel="BM25")

print("Loading MonoT5 Neural Re-ranker...")
monoT5 = MonoT5ReRanker()

# Define Retrieval Strategies
bm25_top3 = bm25_ret % 3
monot5_top3 = bm25_ret % 10 >> monoT5 % 3

# ==========================================
# 3. CUSTOM GENERATIVE READER PIPELINE
# ==========================================
class CustomRAGReader(pt.Transformer):
    def __init__(self, model_name="google/flan-t5-base"):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)

    def transform(self, df):
        results = []
        for qid, group in df.groupby("qid"):
            query = str(group["query"].iloc[0])
            # Gather context from retrieved document IDs
            passages = group.sort_values("rank")["docid"].head(3).tolist()
            context_text = " ".join([str(p) for p in passages])

            prompt = f"Use the context information to answer the Question:\nContext: {context_text}\nQuestion: {query}\nAnswer:"

            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(self.device)
            with torch.no_grad():
                outputs = self.model.generate(**inputs, max_new_tokens=50)
            answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            results.append({"qid": qid, "query": query, "answer": answer})

        return pd.DataFrame(results)

print("Loading Custom Flan-T5 Reader...")
flan_reader = CustomRAGReader("google/flan-t5-base")

# Construct End-to-End RAG Pipelines
bm25_rag_pipeline = bm25_top3 >> flan_reader
monot5_rag_pipeline = monot5_top3 >> flan_reader

# ==========================================
# 4. BENCHMARKING & EVALUATION
# ==========================================
print("Fetching Benchmark Dataset (Vaswani dev set)...")
dev_topics = dataset.get_topics()
dev_topics["query"] = dev_topics["query"].astype(str)

print("Running Pipeline Demo on sample queries...")
sample_topics = dev_topics.head(5).copy()
results = monot5_rag_pipeline.transform(sample_topics)
print("\nSample RAG Generation Results:")
print(results[["qid", "query", "answer"]])

/tmp/ipykernel_1047/2940134330.py:11: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
/tmp/ipykernel_1047/2940134330.py:20: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_ret = pt.BatchRetrieve(sparse_index, wmodel="BM25")


Loading Wikipedia Sparse Index...
Loading MonoT5 Neural Re-ranker...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading Custom Flan-T5 Reader...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Fetching Benchmark Dataset (Vaswani dev set)...
Running Pipeline Demo on sample queries...


InputValidationError: DataFrame does not match required columns for this transformer. [TransformerMode(missing_columns=['text'], extra_columns=[], mode_name=None)]